# 03e — Mức đồng thuận về tầm quan trọng đặc trưng giữa XGBoost, LightGBM và Logistic

**Mục đích:** So sánh mức độ quan trọng của năm nhóm đặc trưng (project_plan mục 3.5e) theo SHAP giữa ba họ mô hình của dự án: LightGBM (mô hình chính), XGBoost và Logistic Regression (Tuần 3 – T2). Phân tích gồm bốn mức: tỷ trọng và thứ hạng nhóm, thứ hạng theo biến, đồng thuận trên từng khách hàng (cơ sở của reason codes) và chiều tác động của biến.

**Thiết lập:**

| Hạng mục | Giá trị |
|---|---|
| Dữ liệu giải thích | 3.000 dòng lấy ngẫu nhiên từ tập Valid (`random_state = 42`), trùng với `03_shap.ipynb` và `03_shap_final_model.ipynb`. Tập Test không được sử dụng |
| LightGBM | `lightgbm_monotone` (`src.tune.fit_tuned_pipeline`), mô hình chính theo `week2_review.md` |
| XGBoost (mặc định) | `xgboost_default`, cấu hình của `03_shap.ipynb` và của bảng so sánh Tuần 2 (ROC-AUC CV 0,7597) |
| XGBoost (`max_depth=4`) | `build_blackbox_pipeline("xgboost")`, cấu hình của `reports/scorecard_vs_blackbox_comparison.md` (ROC-AUC CV 0,788). Dùng để kiểm tra độ nhạy theo mức điều chuẩn của XGBoost |
| Logistic Regression | `logreg_baseline`, SHAP interventional chính xác (`LinearExplainer`), background là tập Train |
| Thang giá trị | Log-odds chưa hiệu chuẩn (margin với mô hình cây, logit với Logistic). Phép so sánh dùng tỷ trọng và thứ hạng nên không phụ thuộc thang |
| Nhóm đặc trưng | `configs/config.yaml`, khóa `feature_groups` |

Cả bốn mô hình dùng chung pipeline tiền xử lý (feature freeze v1) nên có cùng 119 cột đầu vào.

In [ ]:
import sys
import warnings
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.base import clone

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.data_split import load_split_data  # noqa: E402
from src.explain import (  # noqa: E402
    bootstrap_share_ci,
    contribution_agreement,
    effect_direction,
    explain_pipeline,
    group_importance_share,
    load_feature_groups,
    rank_agreement,
    top_group_agreement,
    top_k_overlap,
)
from src.features import build_features  # noqa: E402
from src.pipelines import build_blackbox_pipeline, make_pipeline  # noqa: E402
from src.train import get_advanced_default_models, get_baseline_models  # noqa: E402
from src.tune import fit_tuned_pipeline  # noqa: E402

SEED = 42
N_SAMPLE = 3000
HIGH_RISK_QUANTILE = 0.75   # nhóm rủi ro cao: 25% khách hàng có PD (LightGBM) cao nhất trong mẫu
FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
PREFIX = "shap_agreement"
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)
warnings.filterwarnings("ignore", message="LightGBM binary classifier with TreeExplainer")
print("shap", shap.__version__)

## 1. Huấn luyện và tính SHAP

Bốn mô hình được fit trên toàn bộ Train. Giá trị SHAP của các cột one-hot được cộng về biến gốc rồi về năm nhóm (`ShapResult.by_group`). Tính đúng đắn được kiểm tra bằng sai số cộng tính `base + Σφ = đầu ra mô hình` và, với Logistic Regression, bằng đối chiếu với `shap.LinearExplainer` (background là toàn bộ Train).

In [ ]:
train_df, valid_df, _ = load_split_data()
X_train, y_train = build_features(train_df)
X_valid, y_valid = build_features(valid_df)
X_sample = X_valid.sample(N_SAMPLE, random_state=SEED)

pipelines = {
    "LightGBM": fit_tuned_pipeline("lightgbm_monotone", X_train, y_train),
    "XGBoost (mặc định)": make_pipeline(clone(get_advanced_default_models(SEED)["xgboost_default"])).fit(X_train, y_train),
    "XGBoost (max_depth=4)": build_blackbox_pipeline("xgboost").fit(X_train, y_train),
    "Logistic Regression": make_pipeline(clone(get_baseline_models(SEED)["logreg_baseline"])).fit(X_train, y_train),
}
NAMES = list(pipelines)
groups = load_feature_groups()

results = {
    name: explain_pipeline(pipe, X_sample, background=X_train if name == "Logistic Regression" else None)
    for name, pipe in pipelines.items()
}
by_var = {name: r.by_variable() for name, r in results.items()}
by_group = {name: r.by_group(groups) for name, r in results.items()}
labels = list(by_group[NAMES[0]].columns)
pd_hat = {name: 1 / (1 + np.exp(-r.margin)) for name, r in results.items()}

lr_prep = pipelines["Logistic Regression"].named_steps["preprocess"]
lr_clf = pipelines["Logistic Regression"].named_steps["clf"]
background = shap.maskers.Independent(np.asarray(lr_prep.transform(X_train), dtype=float), max_samples=len(X_train))
lin = shap.LinearExplainer(lr_clf, background).shap_values(np.asarray(lr_prep.transform(X_sample.iloc[:200]), dtype=float))
lin_error = np.abs(lin - results["Logistic Regression"].values[:200]).max()

print("Số cột:", {name: len(r.columns) for name, r in results.items()})
print("Sai số cộng tính lớn nhất:", {name: f"{r.max_additivity_error:.1e}" for name, r in results.items()})
print(f"Sai lệch lớn nhất so với shap.LinearExplainer (200 dòng): {lin_error:.1e}")
assert all(r.max_additivity_error < 1e-4 for r in results.values()) and lin_error < 1e-8
assert all(r.columns == results[NAMES[0]].columns for r in results.values())

## 2. Tỷ trọng và thứ hạng theo nhóm đặc trưng

Tỷ trọng của một nhóm là mean(|SHAP|) của nhóm chia cho tổng năm nhóm, nên tổng mỗi cột bằng 100%. Khoảng tin cậy 95% được tính bằng bootstrap theo khách hàng (500 lần).

In [ ]:
share = pd.DataFrame({name: group_importance_share(by_group[name]) for name in NAMES}).loc[labels]
rank = share.rank(ascending=False).astype(int)
ci = {name: bootstrap_share_ci(by_group[name], n_boot=500, seed=SEED) for name in NAMES}

table = pd.DataFrame(
    {
        name: [
            f"{share.loc[g, name] * 100:.1f}% [{ci[name].loc[g, 'lo'] * 100:.1f}–{ci[name].loc[g, 'hi'] * 100:.1f}] (hạng {rank.loc[g, name]})"
            for g in labels
        ]
        for name in NAMES
    },
    index=labels,
).loc[share[NAMES[0]].sort_values(ascending=False).index]
share.to_csv(FIG_DIR / f"{PREFIX}_group_share.csv")
display(table)

fig, ax = plt.subplots(figsize=(9, 4.5))
width = 0.2
order = list(table.index)
for k, name in enumerate(NAMES):
    ax.bar(np.arange(len(order)) + (k - 1.5) * width, share.loc[order, name] * 100, width, label=name)
ax.set_xticks(range(len(order)), order, rotation=15, ha="right")
ax.set_ylabel("Tỷ trọng mean(|SHAP|) (%)")
ax.set_title("Tầm quan trọng theo nhóm đặc trưng")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIG_DIR / f"{PREFIX}_group_share.png", dpi=150)
plt.show()

## 3. Đồng thuận toàn cục theo cặp mô hình

Với 5 nhóm, Spearman và Kendall tau chỉ nhận một số ít giá trị rời rạc (hoán đổi một cặp hạng kề nhau làm Spearman giảm 0,1), nên được đọc cùng bảng thứ hạng ở mục 2. Hai cột theo biến dùng 49 biến gốc (trước khi cộng nhóm) và cho biết mức đồng thuận có còn khi không gộp nhóm hay không. Cột tương quan PD là tương quan Pearson giữa xác suất vỡ nợ chưa hiệu chuẩn của hai mô hình trên cùng mẫu.

In [ ]:
imp_var = pd.DataFrame({name: by_var[name].abs().mean() for name in NAMES})
rows = []
for a, b in combinations(NAMES, 2):
    g = rank_agreement(share[a], share[b])
    v = rank_agreement(imp_var[a], imp_var[b])
    rows.append({
        "cặp mô hình": f"{a} – {b}",
        "Spearman (nhóm)": g["spearman"],
        "Kendall (nhóm)": g["kendall"],
        "Spearman (biến)": v["spearman"],
        "Trùng Top 10 biến": top_k_overlap(imp_var[a], imp_var[b], 10),
        "Tương quan PD": float(np.corrcoef(pd_hat[a], pd_hat[b])[0, 1]),
    })
pairwise = pd.DataFrame(rows).set_index("cặp mô hình")
pairwise.to_csv(FIG_DIR / f"{PREFIX}_pairwise_global.csv")
print("Số biến gốc:", imp_var.shape[0])
pairwise.round(3)

## 4. Đồng thuận trên từng khách hàng (cơ sở của reason codes)

Reason codes dựa trên đóng góp của từng nhóm cho từng khách hàng, nên mức đồng thuận toàn cục chưa đủ để kết luận hai mô hình đưa ra cùng lý do. Hai chỉ số bổ sung:

- **Spearman theo nhóm:** tương quan hạng giữa đóng góp SHAP của một nhóm ở hai mô hình, tính qua 3.000 khách hàng.
- **Trùng reason code đầu tiên:** tỷ lệ khách hàng có cùng nhóm làm tăng rủi ro mạnh nhất. Chỉ số được báo cáo trên toàn mẫu và trên nhóm rủi ro cao (25% khách hàng có PD cao nhất theo LightGBM, định nghĩa theo phân vị để không phụ thuộc ngưỡng cảnh báo chưa chốt).

In [ ]:
high_risk = pd_hat["LightGBM"] >= np.quantile(pd_hat["LightGBM"], HIGH_RISK_QUANTILE)
rows = []
for a, b in combinations(NAMES, 2):
    row = {"cặp mô hình": f"{a} – {b}"}
    row.update({f"ρ {g}": v for g, v in contribution_agreement(by_group[a], by_group[b]).items()})
    row["Trùng reason code đầu (toàn mẫu)"] = top_group_agreement(by_group[a], by_group[b])
    row["Trùng reason code đầu (rủi ro cao)"] = top_group_agreement(by_group[a], by_group[b], mask=high_risk)
    rows.append(row)
per_customer = pd.DataFrame(rows).set_index("cặp mô hình")
per_customer.to_csv(FIG_DIR / f"{PREFIX}_per_customer.csv")
print(f"Nhóm rủi ro cao: {int(high_risk.sum())}/{len(high_risk)} khách hàng")
per_customer.round(2).T

## 5. Chiều tác động của biến

Chiều tác động là Spearman giữa giá trị biến và đóng góp SHAP của chính biến đó (dương: giá trị càng cao càng làm tăng rủi ro). Mô hình Logistic có quan hệ đơn điệu theo cấu trúc (|ρ| = 1); với mô hình cây, giá trị gần ±1 cho thấy quan hệ gần đơn điệu, giá trị gần 0 cho thấy quan hệ không đơn điệu. LightGBM bị ràng buộc đơn điệu tăng đối với `PAY_MAX`, `PAY_MEAN`, `PAY_LATE_2PLUS_COUNT` và `UTIL_MEAN` (`configs/config.yaml`, khóa `tuning.monotone_increasing`).

In [ ]:
frame = pipelines["LightGBM"].named_steps["preprocess"][:-1].transform(X_sample)
VARIABLES = ["PAY_MAX", "PAY_MEAN", "PAY_LATE_2PLUS_COUNT", "UTIL_MEAN", "LIMIT_BAL",
             "BILL_AMT1", "BILL_DELTA", "PAY_AMT1", "PAY_RATIO_MEAN"]
direction = pd.DataFrame({name: effect_direction(by_var[name], frame, VARIABLES) for name in NAMES})
direction.to_csv(FIG_DIR / f"{PREFIX}_effect_direction.csv")
direction.round(2)

## 6. Nhận xét

### 6.1. Tính đúng đắn

- Sai số cộng tính lớn nhất là 3,3·10⁻⁶ (XGBoost mặc định) và nhỏ hơn ở ba mô hình còn lại; công thức tuyến tính chính xác cho Logistic Regression trùng với `shap.LinearExplainer` (sai lệch 0).
- Platt scaling (Tuần 3 – T2 của B) nhân mọi đóng góp SHAP với cùng một hệ số `a` (≈ 1,01), nên tỷ trọng và thứ hạng nhóm ở các bảng trên không thay đổi sau khi hiệu chuẩn.

### 6.2. Đồng thuận toàn cục theo nhóm

- **Hai đầu bảng xếp hạng ổn định ở cả bốn mô hình.** Lịch sử trễ hạn luôn hạng 1 (38,5% đến 55,7%) và nhân khẩu học luôn hạng 5 (2,3% đến 7,4%).
- **Ba nhóm giữa không thống nhất về thứ tự.** LightGBM xếp mức sử dụng hạn mức (18,7%), dư nợ (16,4%), hành vi trả nợ (13,5%). XGBoost `max_depth=4` và Logistic cùng xếp mức sử dụng hạn mức, hành vi trả nợ, dư nợ (Spearman nhóm = 1,0 giữa hai mô hình này). XGBoost mặc định xếp hành vi trả nợ ở hạng 2 với 24,1%, gần gấp đôi LightGBM (13,5%).
- Các khoảng tin cậy bootstrap hẹp (rộng không quá 2 điểm phần trăm), nên chênh lệch tỷ trọng giữa các mô hình không do sai số lấy mẫu mà do bản chất mô hình.
- Logistic Regression gán cho nhân khẩu học 7,4%, cao hơn các mô hình cây (2,3% đến 5,3%).

### 6.3. Ảnh hưởng của mức điều chuẩn XGBoost

- Khác biệt giữa XGBoost mặc định và LightGBM phần lớn đến từ siêu tham số chứ không từ thuật toán. XGBoost `max_depth=4` gần LightGBM hơn nhiều: Spearman theo biến 0,92 (mặc định: 0,73), trùng Top 10 biến 8/10 (mặc định: 4/10), tương quan PD 0,992 (mặc định: 0,897).
- XGBoost mặc định vi phạm chiều tác động nghiệp vụ ở hai biến mà LightGBM bị ràng buộc đơn điệu: `UTIL_MEAN` (−0,33, tức mức sử dụng hạn mức cao làm giảm đóng góp rủi ro) và `PAY_MEAN` (−0,04, không có chiều rõ ràng). LightGBM giữ chiều dương ở cả bốn biến bị ràng buộc (0,68 đến 0,81).
- Kết luận về đồng thuận vì vậy phụ thuộc vào cấu hình XGBoost được chọn làm đại diện. Việc XGBoost không vào nhóm mô hình cuối (`week2_review.md`) dựa trên cấu hình mặc định; bản `max_depth=4` có ROC-AUC CV khoảng 0,788, gần LightGBM, nhưng chưa được tuning hay đăng ký trong registry.

### 6.4. Logistic Regression và mô hình cây

- **Đồng thuận ở mức nhóm nhưng không ở mức biến.** Spearman nhóm giữa Logistic và LightGBM là 0,9, trong khi Spearman theo biến chỉ 0,30 (với XGBoost mặc định: 0,02) và chỉ 6/10 biến trùng Top 10.
- **Nguyên nhân khả dĩ là đa cộng tuyến trong Logistic** (chưa được kiểm định riêng trong notebook này). Chiều tác động của một số biến đảo ngược so với mô hình cây: `PAY_MEAN` (−1,0 so với +0,79), `BILL_AMT1` (+1,0 so với −0,66) và `PAY_RATIO_MEAN` (+1,0 so với −0,81). Các biến trễ hạn, dư nợ và mức sử dụng hạn mức tương quan cao với nhau (project_plan mục 3.5e), nên hệ số Logistic có thể bù trừ lẫn nhau và đổi dấu.
- **Đóng góp của nhóm dư nợ và biến động dư nợ tương quan âm giữa Logistic và các mô hình cây** (Spearman theo khách hàng −0,62 với LightGBM, −0,51 với XGBoost `max_depth=4`, −0,40 với XGBoost mặc định). Đây là nhóm có mức đồng thuận thấp nhất.
- Nhóm lịch sử trễ hạn đồng thuận cao ở mọi cặp (Spearman theo khách hàng 0,86 đến 0,98). Mức sử dụng hạn mức ở mức khá (0,63 đến 0,95); hành vi trả nợ (0,45 đến 0,95) và nhân khẩu học (0,53 đến 0,88) thấp hơn.

### 6.5. Đồng thuận reason code đầu tiên

- Trên nhóm rủi ro cao (750/3.000 khách hàng), nhóm làm tăng rủi ro mạnh nhất của LightGBM trùng với XGBoost `max_depth=4` ở 96% khách hàng, với Logistic ở 76% và với XGBoost mặc định ở 74%.
- Trên toàn mẫu, tỷ lệ trùng thấp hơn (88%, 49% và 66%). Khách hàng rủi ro thấp có đóng góp dương rất nhỏ, nên nhóm "mạnh nhất" ở nhóm này ít ý nghĩa và dễ khác nhau giữa các mô hình.

### 6.6. Hệ quả cho reason codes và demo

- Reason codes theo nhóm đáng tin ở nhóm lịch sử trễ hạn, nhóm chi phối trường hợp Cảnh báo cao được minh họa tại `03_shap_local.ipynb`, và nhất quán giữa các họ mô hình. Với nhóm dư nợ và biến động dư nợ, hướng đóng góp phụ thuộc mô hình, nên không nên diễn giải riêng nhóm này như một nguyên nhân độc lập.
- Reason codes theo biến không nên dùng trên Logistic Regression vì dấu hệ số không ổn định do đa cộng tuyến. Điều này củng cố lựa chọn LightGBM (có ràng buộc đơn điệu) làm mô hình giải thích trên demo.

### 6.7. Giới hạn

- Mẫu giải thích gồm 3.000 dòng của tập Valid. Với năm nhóm, Spearman và Kendall chỉ nhận số ít giá trị và nhạy với một cặp hạng đổi chỗ; các chỉ số theo khách hàng và theo biến cung cấp thông tin bổ sung.
- Hai họ mô hình dùng hai định nghĩa kỳ vọng nền khác nhau (`TreeExplainer` theo `tree_path_dependent`, Logistic theo background Train), nên base value không so sánh trực tiếp; phép so sánh chỉ dùng tỷ trọng, thứ hạng và tương quan.
- Chiều tác động (Spearman giữa giá trị biến và đóng góp SHAP) mô tả quan hệ trung bình trong mô hình, không phải quan hệ nhân quả, và bị ảnh hưởng bởi tương quan giữa các biến.

## Hình và bảng xuất ra

Lưu tại `reports/figures/` (không commit, sinh lại bằng cách chạy notebook), tiền tố `shap_agreement_`:

- `_group_share.png`, `_group_share.csv` — tỷ trọng theo nhóm của bốn mô hình.
- `_pairwise_global.csv` — đồng thuận toàn cục theo cặp.
- `_per_customer.csv` — đồng thuận trên từng khách hàng.
- `_effect_direction.csv` — chiều tác động của biến.